# Line Predictors
### Notebook initialisation

In [ ]:
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)

import torch, random, os, cv2
import numpy as np

seed = 1

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(seed)

import seaborn as sns
sns.set_theme(style="whitegrid", context="paper")

### Dataset Loading

In [ ]:
robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"

from pose_estimation.headset_data import HeadsetData, create_robot_bound_headset_data
from pose_estimation.robot_environment import RobotEnvironment, GatheredRobotData, visualize_robot_camera_environment_combo, XYZImageGenerationConfig, ICPAlignmentConfig

robot_data = GatheredRobotData.from_folder(robot_data_folder_location)
robot_env = RobotEnvironment.from_gathered_robot_data(
        robot_data = robot_data,
        number_of_sampled_datapoints=10,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=False)
)

labeled_headset_data = create_robot_bound_headset_data(
        headset_data = HeadsetData.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

visualize_loaded_data = False

if visualize_loaded_data:
    visualize_robot_camera_environment_combo(robot_env=robot_env, headset_data=labeled_headset_data)

## Hyperparameters
### Simple Predictor

In [ ]:
from pose_estimation.pose_pred_points import OnlyPointsPredictor, ExtractAndMatchWrapperConfig
from pose_estimation.pose_pred_points_lines import LinePredictor
from pose_estimation.predictor_grader import PredictionOnDataset

default_point_predictor = LinePredictor(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig()
)

PredictionOnDataset(predictor = default_point_predictor,headset_data = labeled_headset_data).print_summary()

### Creating Predictors
Now an `PosePredictor` can be created. An `PosePredictor` instance is build upon an `RobotEnvironment` instance and can predict positions from headset-images.

In [ ]:
predictor = LinePredictor(
    cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
    cam1_bgr_images=robot_env.robot_bgr_images,
    cam1_xyz_images=robot_env.robot_xyz_images,
    extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
        extract_and_match=ExtractAndLightGlue(
            extractor="SuperPoint"
        ),
        ransac_config=pose_estimation_ransaac_config_less_precise,
    ),
    line_fitting_3d_config=LineFitting3dConfig(use_ransac=False),
    pnpl_optimisation_conf=PnPLOptimizerConfig(lm_max_steps=10000,line_relevance=1.0),
    cam2_lsd_diagonal_size=None,
    debug_visualize_line_cleanup=False, 
    debug_visualize_pnpl=False
)

### Grading the Performance of an Initialised Predictor:

In [ ]:
init_predictor_grade = PredictionOnDataset(
    predictor = predictor,
    headset_data = labeled_headset_data,
    number_retry = 1
)
init_predictor_grade.print_summary()


visualize_prediction = True
if visualize_prediction:
    init_predictor_grade.visualize_predictions(
        robot_env=robot_env,
        show_label=False
    )

In [ ]:
# Creation of the Predictors
light_glue = ExtractAndLightGlue()


point_based = GradablePosePredictor(
    creator=OnlyPointsPredictor.get_creation_function(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Rotate180Deg],
            extract_and_match=light_glue,
            ransac_config=pose_estimation_ransaac_config_precise,
        )
    ),
    name="only points"
)

line_predictor = GradablePosePredictor(
    creator=LinePredictor.get_creation_function(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndLightGlue(
                extractor="SuperPoint"
            ),
            ransac_config=pose_estimation_ransaac_config_less_precise,
        ),
        line_fitting_3d_config=LineFitting3dConfig(use_ransac=False),
        pnpl_optimisation_conf=PnPLOptimizerConfig(lm_max_steps=10000,line_relevance=0.3),
        cam2_lsd_diagonal_size=1000,
        debug_visualize_line_cleanup=False, 
        debug_visualize_pnpl=False 
    ),
    name="lines"
)

Now those can be used to create a grader object for multiple `PosePredictor` variants.

In [ ]:
# Initialising the grader:
grader = NPredictors1DatasetGrader(
    gradable_pose_predictors=[point_based, line_predictor],
    headset_data = labeled_headset_data,
    robot_env = robot_env,
)

In [ ]:
visualize_trajectories_3d = False
if visualize_trajectories_3d:
    grader.visualize_predictions_3d()

grader.print_summary()

fig1, ax1 = plt.subplots(1, 1, figsize = (16, 8))
grader.plot_translational_errors(ax1)

fig2, ax2 = plt.subplots(1, 1, figsize = (16, 8))
grader.plot_rotational_errors(ax2)

fig3, ax3 = plt.subplots(1, 1, figsize = (16, 8))
grader.plot_creation_times(ax3)

fig4, ax4 = plt.subplots(1, 1, figsize = (8, 5))
grader.plot_successful_frame_prediction_times(ax4)


plt.show()

### Video visualisation

In [ ]:
video_predictor = LinePredictor(
    cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
    cam1_bgr_images=robot_env.robot_bgr_images,
    cam1_xyz_images=robot_env.robot_xyz_images,
    extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
        extract_and_match=ExtractAndLightGlue(
            extractor="SuperPoint"
        ),
        ransac_config=pose_estimation_ransaac_config_less_precise,
    ),
    line_fitting_3d_config=LineFitting3dConfig(use_ransac=False),
    pnpl_optimisation_conf=PnPLOptimizerConfig(lm_max_steps=10000,line_relevance=0.3),
    cam2_lsd_diagonal_size=None,
    debug_visualize_line_cleanup=False, 
    debug_visualize_pnpl=False
)

init_predictor_grade = PredictionOnDataset(
    predictor = video_predictor,
    headset_data = labeled_headset_data,
    number_retry = 1,
    vid_gen=VideoGenerator(fps=20)
)
init_predictor_grade.print_summary()